# Самостоятельный эксперимент: learning_rate_3e4
Начальный learning_rate: 0.001 → 0.0003; 512 сред, 1001 итерация. Гипотеза записана до запуска в обучающей ячейке. Полные результаты будут добавлены после завершения.

Обучение завершено: 1001 итерация, Mean reward последней итерации 673.91, время 1122 с (18.7 мин). Инференс: SR 0/10, средняя награда 691.71.


In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
%pip uninstall -y jax-cuda13-plugin jax-cuda13-pjrt
%pip install -q "playground[learning]==0.1.0" "jax[cuda12]==0.6.2" "mujoco==3.5.0" "mujoco-mjx==3.5.0" tensorboard pytest
import os, subprocess, sys
os.chdir("/content")
os.environ["MUJOCO_GL"] = "egl"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
subprocess.run([sys.executable, "-c", 'import jax, torch; print("JAX:", jax.devices()); print("Torch CUDA:", torch.cuda.is_available()); assert jax.default_backend() == "gpu" and torch.cuda.is_available()'], check=True)

Found existing installation: jax-cuda13-plugin 0.11.1
Uninstalling jax-cuda13-plugin-0.11.1:
  Successfully uninstalled jax-cuda13-plugin-0.11.1
Found existing installation: jax-cuda13-pjrt 0.11.1
Uninstalling jax-cuda13-pjrt-0.11.1:
  Successfully uninstalled jax-cuda13-pjrt-0.11.1

CompletedProcess(args=['/usr/bin/python3', '-c', 'import jax, torch; print("JAX:", jax.devices()); print("Torch CUDA:", torch.cuda.is_available()); assert jax.default_backend() == "gpu" and torch.cuda.is_available()'], returncode=0)

In [ ]:
%%script python3 -u
print("СКРИПТ ЗАПУЩЕН", flush=True)
"""Встроенные config.py + train.py для Colab (без внешнего репозитория)."""
print("Начинаю импорт библиотек...", flush=True)
import json
import os
import time
from datetime import datetime
from types import SimpleNamespace
import jax
import torch
from mujoco_playground import registry, wrapper_torch
from mujoco_playground.config import manipulation_params
from rsl_rl.runners import OnPolicyRunner
jax.config.update("jax_default_matmul_precision", "highest")
# --- Параметры запуска (аналог аргументов CLI из train.py) ---
NUM_ENVS = 512
MAX_ITERS = 1001
SAVE_INTERVAL = 50
SEED = 1
EXP_NAME = "learning_rate_3e4"
RESUME = None  # путь к чекпоинту или None
DEVICE = "cuda:0"
# --- config.py ---
ENV_NAME = "PandaPickCube"
ACTOR_HIDDEN_DIMS = [512, 256, 128]
CRITIC_HIDDEN_DIMS = [512, 256, 128]
ACTIVATION = "elu"
INIT_NOISE_STD = 1.0
def build_runner_cfg(obs_size) -> dict:
    cfg = manipulation_params.rsl_rl_config(ENV_NAME).to_dict()
    _policy = cfg.pop("policy", {})
    cfg["actor"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": ACTOR_HIDDEN_DIMS,
        "activation": ACTIVATION,
        "distribution_cfg": {
            "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
            "init_std": INIT_NOISE_STD,
        },
    }
    cfg["critic"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": CRITIC_HIDDEN_DIMS,
        "activation": ACTIVATION,
    }
    if isinstance(obs_size, dict):
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["privileged_state"]}
    else:
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["state"]}
    return cfg
def resolve_log_dir(exp_name, resume) -> str:
    if exp_name:
        name = exp_name
    elif resume:
        name = os.path.basename(os.path.dirname(os.path.abspath(resume)))
    else:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = f"{ENV_NAME}_{timestamp}"
    log_dir = os.path.join("logs", name)
    os.makedirs(log_dir, exist_ok=True)
    return log_dir
args = SimpleNamespace(
    exp_name=EXP_NAME,
    num_envs=NUM_ENVS,
    max_iters=MAX_ITERS,
    device=DEVICE,
    resume=RESUME,
    save_interval=SAVE_INTERVAL,
    seed=SEED,
)
device = args.device
device_rank = int(device.split(":")[-1]) if "cuda" in device else 0
log_dir = resolve_log_dir(args.exp_name, args.resume)
print(f"Среда:            {ENV_NAME}")
print(f"Устройство:       {device}")
print(f"Параллельных сред: {args.num_envs}")
print(f"Итерации:         {args.max_iters}")
print(f"Лог/чекпоинты:    {log_dir}")
print()
env_cfg = registry.get_default_config(ENV_NAME)
raw_env = registry.load(ENV_NAME, config=env_cfg)
brax_env = wrapper_torch.RSLRLBraxWrapper(
    raw_env,
    num_actors=args.num_envs,
    seed=args.seed,
    episode_length=env_cfg.episode_length,
    action_repeat=1,
    device_rank=device_rank,
)
brax_env.cfg = {}
cfg_dict = build_runner_cfg(raw_env.observation_size)
cfg_dict["algorithm"]["learning_rate"] = 0.0003
print("Начальный learning_rate:", cfg_dict["algorithm"]["learning_rate"])
cfg_dict["seed"] = args.seed
cfg_dict["max_iterations"] = args.max_iters
cfg_dict["save_interval"] = args.save_interval
config_path = os.path.join(log_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(cfg_dict, f, indent=2, default=str)
print(f"Конфиг сохранён: {config_path}")
runner = OnPolicyRunner(brax_env, cfg_dict, log_dir, device=device)
if args.resume:
    print(f"Загрузка чекпоинта: {args.resume}")
    runner.load(args.resume)
print("=" * 60)
print("  Начинаю обучение...")
print("=" * 60)
print()
t_start = time.time()
runner.learn(
    num_learning_iterations=args.max_iters,
    init_at_random_ep_len=False,
)
elapsed = time.time() - t_start
print()
print("=" * 60)
print(f"  Обучение завершено за {elapsed:.0f} с ({elapsed / 60:.1f} мин)")
print(f"  Логи и чекпоинты: {log_dir}")
print("=" * 60)

СКРИПТ ЗАПУЩЕН
Начинаю импорт библиотек...
Среда:            PandaPickCube
Устройство:       cuda:0
Параллельных сред: 512
Итерации:         1001
Лог/чекпоинты:    logs/learning_rate_3e4

mujoco_menagerie not found. Downloading...
Cloning mujoco_menagerie: ██████████| 100/100 [00:49<00:00]
Checking out commit 1b86ece576591213e2b666ebf59508454200ca97
Successfully downloaded mujoco_menagerie
Device -- cuda:0
Key device -- {CudaDevice(id=0)}
obs_shape: 66
JITing reset and step
Done JITing reset and step
Начальный learning_rate: 0.0003
Конфиг сохранён: logs/learning_rate_3e4/config.json
--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ['state']
	 critic :  ['state']
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): Identity()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=66, out_features=512, bias=True)
    (1)

In [ ]:
# Скачивание результатов самостоятельного опыта
from pathlib import Path
import shutil
from google.colab import files
run_dir = Path("/content/logs/learning_rate_3e4")
checkpoints = sorted(run_dir.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
assert checkpoints, "Чекпоинты не найдены — проверьте вывод обучения"
print("Последний чекпоинт:", checkpoints[-1].name)
archive = shutil.make_archive("/content/learning_rate_3e4", "zip", "/content/logs", "learning_rate_3e4")
print("Архив опыта:", archive)
files.download(archive)

Последний чекпоинт: model_1000.pt
Архив опыта: /content/learning_rate_3e4.zip


Инференс learning_rate_3e4: захваты 0/10, SR 0/10, средняя награда 691.71. Проверено 10 эпизодов на CPU без записи видео. Визуальное поведение не описано. Снижение начального learning_rate не улучшило успешность в этой проверке. Гипотеза о близкой к baseline награде и отсутствии улучшения SR согласуется с наблюдением; по одному обучению и 10 эпизодам нельзя установить общий эффект параметра.